# D0: frozen representation–VQC diagnostics

## Objective
Explore completed D0-A/D0-B diagnostics for five development folds and seeds 1729–1731. No training, calibration, raw cohort access, or checkpoint loading occurs in this notebook. **Representation variation, functional sensitivity, and predictive effects are different quantities.** Large sensitivity is not useful information; small sensitivity is not wasted information.

The [self-contained report](../reports/phase2/d0_v1_verified/report.md) states numerical results and limitations. The [approved protocol](../reports/phase2/d0_protocol_v1.md) records the fixed design. The [CV notebook](02_development_cross_validation.ipynb) already covers original predictive results, generalization gaps, learning curves, saturation, and latent spectra.

Observed D0 results: B's mean validation variance entropy rank is 1.4402 in z and 1.5235 after the map. Ten nearly constant coordinates across nine runs have map slopes near their maximum but low local circuit response. Mean replacement is essentially inert there; zero replacement changes probabilities and reduces log loss in these artificial interventions. This does not establish a beneficial role for the constants or justify a training change.

## Load existing artifacts
The default input is the verified SVD-based D0 summary directory. Set `QARL_D0_ANALYSIS_DIR` to relocate it. All patient-level derivatives and intervention predictions remain private under `runs/`; this notebook needs only aggregate CSVs. Run from the repository root or `notebooks/`, using the existing analysis/notebook dependencies.

In [ ]:
import os
from pathlib import Path
import sys
import matplotlib.pyplot as plt
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if not (root / "src/phase2").is_dir():
    raise RuntimeError("Run from the repository root or notebooks directory.")
sys.path.insert(0, str(root))
from src.phase2.d0_views import (load_tables, plot_geometry, plot_decomposition,
    plot_shared_sensitivity, plot_interventions, plot_constant_ablation,
    plot_support, plot_local_approximation)
analysis = Path(os.environ.get("QARL_D0_ANALYSIS_DIR", str(root / "reports/phase2/d0_v1_verified")))
tables = load_tables(analysis)
plt.rcParams.update({"font.size": 10, "figure.dpi": 110,
                     "axes.spines.top": False, "axes.spines.right": False})
# These select an existing run for detailed display; they do not select a model.
fold, seed = 0, 1729


## Diagnostics and visualizations
### Numerical checks
The angular derivative includes π: dφ/dz=(π/2)(1−tanh²(z/2)). Direct df/dz is compared with (df/dφ)(dφ/dz) on every saved patient. Probability finite differences use the first 19 canonical-order patients of each split. The redundant final RZ control is retained. Tolerances validate calculations, not discoveries.

In [ ]:
display(tables["checks"][["fd_probability_max_abs_error", "chain_max_abs_error",
    "final_rz_perturbation_max_abs_error"]].max().to_frame("Maximum absolute error"))

### Representation variation
Each line is one fold/seed. Entropy rank uses centered variance weights. A change between z and φ(z) does not demonstrate a training cause or information loss. B_classical's angular transform is hypothetical: its actual head does not use it.

In [ ]:
plot_geometry(tables["geometry"])
plt.show()

### Separate map slope from circuit response
The detailed panel uses the configurable fold/seed above. Coordinate indices belong only to that run. Map slope relative to π/2 describes attenuation relative to its maximum; it is not directly comparable with circuit derivatives or coordinate SD. The unnormalized map slope may exceed one, and derivative norms use different input units.

In [ ]:
plot_decomposition(tables["coordinates"], fold=fold, seed=seed)
plt.show()
part = tables["coordinates"]
display(part[(part.fold == fold) & (part.seed == seed) & (part.model == "B") &
             (part.checkpoint == "final") & (part.split == "validation")][[
    "coordinate", "training_sd", "map_derivative_relative_max_mean",
    "angle_expectation_gradient_rms", "z_expectation_gradient_rms"]])

### B and its probe: shared representation, different local responses
Each line pairs the same fold/seed and embedding. Raw and training-SD-scaled gradients describe different input units, without establishing utility. The cosine table retains exceptions; low alignment is not evidence that one head uses more valuable information. A sigmoid linear head has collinear probability gradients, so its sensitivity entropy rank of one is an algebraic property.

In [ ]:
plot_shared_sensitivity(tables["functional"])
plt.show()
shared = tables["shared_scaled"]
display(shared[shared.split == "validation"].groupby("scale")["cosine_mean"].agg(["mean", "min", "max"]))

### Predictive effects of frozen interventions
Effects are altered minus original metrics; negative log-loss changes are reductions. Each dot averages the fixed intervention cells within one fold/seed. Local panels show α=0.1; all α values remain in the tables below. B/probe receive identical altered z. B_classical has an independently learned representation; only aggregate effects are compared, with no coordinate correspondence.

Mean and zero replacement act differently for B. For a linear logistic head, those two replacements differ by a constant logit offset for each coordinate, so identical ROC-AUC changes are expected. Uniform local translations also preserve linear-head ranking. These are structural controls, not evidence of globally superior robustness.

In [ ]:
plot_interventions(tables["intervention_summary"])
plt.show()
effects = tables["effect_distribution"]
display(effects[(effects.model == "B") & (effects.split == "validation") &
               effects.measure.isin(["delta_roc_auc_mean", "delta_log_loss_mean"])][[
    "family", "alpha", "measure", "mean", "between_fold_sd", "mean_within_fold_seed_sd", "minimum", "maximum"]].round(6))

### Nearly constant coordinates: offset versus variation
The continuous x-axis avoids a discovery threshold. Display floors are explicitly labeled. A separate table uses relative training SD <1e−6 solely as a numerical description of the previously observed contraction, not a recurrence or discovery criterion. Mean replacement of those B coordinates changes mean absolute probabilities by at most 3.44e−13; zero replacement produces larger shifts. Zero replacement reduces B's validation log loss for all ten coordinate cases, but places every altered patient outside that coordinate's training range. This does not establish useful constants or a recommended intervention.

In [ ]:
plot_constant_ablation(tables["coordinate_ablation"])
plt.show()
a = tables["coordinate_ablation"]
small = a[(a.model == "B") & (a.split == "validation") & (a.relative_training_sd < 1e-6)]
display(small[["fold", "seed", "coordinate", "family", "training_sd",
    "probability_change_abs_mean", "delta_roc_auc", "delta_log_loss", "outside_training_bounds_after"]])

### Effective displacement and limited support checks
Scales, axes, means, and donor values come only from training. Validation donors are sampled from training without replacement; this is not a within-validation permutation. Bounds and nearest-neighbor distance are limited descriptors: recombined inputs can remain inside marginal ranges while departing from the joint distribution. B/probe have the same displacements, so the plot shows B once. B_classical is displayed only at aggregate level.

In [ ]:
plot_support(tables["intervention_summary"])
plt.show()

### Local approximation and finite changes
These curves compare actual probability shifts with the first-order prediction, separately from predictive loss. Mean absolute error falls with α. A weak local derivative does not imply invariance under a larger displacement; zero replacement can probe curvature and out-of-distribution regions. No perturbation was chosen for its validation effect.

In [ ]:
plot_local_approximation(tables["intervention_summary"])
plt.show()

## Interpretation
H1 is weakened as a general explanation based on angular saturation: nearly constant coordinates have map slopes near the maximum, and B's angular entropy rank is higher on average. Local attenuation exists elsewhere but does not identify a training cause. H2 is supported only as a description of uneven local response; useful ignored information is not established. H3 supports dependence on constant input levels in frozen solutions, but the claim that their learned values are beneficial is weakened. H4 is descriptively supported: B and the probe have different local responses and intervention effects on the same embedding.

## Limitations and next experimental questions
These are exploratory dependent fold/seed results, not independent replications or causal training interventions. Intermediate checkpoints are unavailable. Initial/final differences change both representation and head, so they do not identify which caused the final geometry. Local response, statistical variation, and predictive effects remain distinct. No C0/C1 method, definitive regularizer, head refit, calibration, architecture change, or locked-test evaluation was performed. Separate experiments would be needed to test alternative heads on the same frozen encoder, angular-map effects during training, encoder-gradient controls, and offset/calibration mechanisms. Prior-art assessment is required before an original-method proposal.